# Historical Stock Prices and Revenue

A comparison of Tesla and GameStop using Yahoo Finance price history and archived quarterly revenue tables from the IBM Python Project for Data Science course. Prices are plotted through 14 June 2021 and revenue through 30 April 2021, matching the original exercise. The two sources cover different date ranges.

Run the cells in order after installing `requirements.txt`. Downloads require an internet connection.

In [ ]:
import pandas as pd
import requests
import yfinance as yf
from bs4 import BeautifulSoup
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [ ]:
def load_stock(ticker):
    data = yf.Ticker(ticker).history(period="max", auto_adjust=False, timeout=30)
    if data.empty:
        raise ValueError(f"No price data returned for {ticker}. Check the connection and try again.")
    data = data.reset_index()
    data["Date"] = pd.to_datetime(data["Date"]).dt.tz_localize(None)
    return data


def parse_revenue(html):
    soup = BeautifulSoup(html, "html.parser")
    table = next(
        (table for table in soup.find_all("table")
         if "quarterly revenue" in table.get_text(" ", strip=True).lower()),
        None,
    )
    if table is None:
        raise ValueError("The page does not contain a quarterly revenue table.")

    rows = []
    for row in table.find_all("tr"):
        columns = row.find_all("td")
        if len(columns) == 2:
            rows.append([column.get_text(strip=True) for column in columns])

    data = pd.DataFrame(rows, columns=["Date", "Revenue"])
    data["Date"] = pd.to_datetime(data["Date"], format="%Y-%m-%d", errors="coerce")
    amounts = data["Revenue"].str.replace(r"[$,]", "", regex=True)
    data["Revenue"] = pd.to_numeric(amounts, errors="coerce")
    data = data.dropna().sort_values("Date").reset_index(drop=True)
    if data.empty:
        raise ValueError("The quarterly revenue table has no usable rows.")
    return data


def load_revenue(url):
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    return parse_revenue(response.text)


def make_graph(stock_data, revenue_data, stock):
    prices = stock_data[stock_data["Date"] <= pd.Timestamp("2021-06-14")].sort_values("Date")
    revenue = revenue_data[revenue_data["Date"] <= pd.Timestamp("2021-04-30")].sort_values("Date")
    if prices.empty or revenue.empty:
        raise ValueError(f"No {stock} data is available within the chart's date range.")

    fig = make_subplots(
        rows=2, cols=1, shared_xaxes=True,
        subplot_titles=("Historical Share Price", "Quarterly Revenue"),
    )
    fig.add_trace(go.Scatter(x=prices["Date"], y=prices["Close"], name="Share Price"), row=1, col=1)
    fig.add_trace(go.Scatter(x=revenue["Date"], y=revenue["Revenue"], name="Revenue"), row=2, col=1)
    fig.update_xaxes(title_text="Date", row=2, col=1)
    fig.update_yaxes(title_text="Closing price (USD)", row=1, col=1)
    fig.update_yaxes(title_text="Revenue (USD millions)", row=2, col=1)
    fig.update_layout(title=stock, height=900, showlegend=False)
    return fig

## 1. Tesla price history

In [ ]:
tesla_data = load_stock("TSLA")
tesla_data.head()

## 2. Tesla quarterly revenue

In [ ]:
tesla_url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-PY0220EN-SkillsNetwork/labs/project/revenue.htm"
tesla_revenue = load_revenue(tesla_url)
tesla_revenue.tail()

## 3. GameStop price history

In [ ]:
gme_data = load_stock("GME")
gme_data.head()

## 4. GameStop quarterly revenue

In [ ]:
gme_url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-PY0220EN-SkillsNetwork/labs/project/stock.html"
gme_revenue = load_revenue(gme_url)
gme_revenue.tail()

## 5. Tesla charts

In [ ]:
tesla_figure = make_graph(tesla_data, tesla_revenue, "Tesla")
tesla_figure.show()

## 6. GameStop charts

In [ ]:
gme_figure = make_graph(gme_data, gme_revenue, "GameStop")
gme_figure.show()